In [4]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm.notebook import tqdm

luxury_prompts = [
    "luxury interior design, expensive marble textures, gold accents",
    "high-end designer furniture, velvet sofas, crystal chandelier",
    "panoramic windows, modern aesthetic, warm elegant lighting",
    "premium renovation, natural wood, leather details, spacious",
    "sophisticated color palette, beige and gold tones, minimalist luxury"
]

standard_prompts = [
    "standard apartment renovation, simple furniture, plastic materials",
    "cheap interior, poor lighting, messy, old wallpaper",
    "basic renovation, standard windows, dull colors"
]

def get_luxury_score(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        
        text_inputs = processor(text=luxury_prompts + standard_prompts, return_tensors="pt", padding=True).to(device)
        image_inputs = processor(images=image, return_tensors="pt").to(device)
        
        with torch.no_grad():
            image_features = model.get_image_features(**image_inputs)
            text_features = model.get_text_features(**text_inputs)

            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            luxury_scores = (image_features @ text_features.T)[0, :len(luxury_prompts)]
            standard_scores = (image_features @ text_features.T)[0, len(luxury_prompts):]

            score = luxury_scores.mean() - standard_scores.mean()
            return score.item()
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return 0.0

In [5]:
# code from hf
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=["a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
DATA_DIR = "/kaggle/input/photos"

results = []

folders = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]

In [ ]:
for folder_id in tqdm(folders):
    folder_path = os.path.join(DATA_DIR, folder_id)
    file_names = os.listdir(folder_path)
    
    image_files = [f for f in file_names if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if len(image_files) == 0:
        continue
        
    folder_scores = []
    
    for img_name in image_files:
        img_path = os.path.join(folder_path, img_name)
        score = get_luxury_score(img_path)
        folder_scores.append(score)
    
    max_score = max(folder_scores)
    
    results.append({"id": folder_id, "luxury_score": max_score})

In [ ]:
df = pd.DataFrame(results)
df = df.sort_values(by="id")

In [13]:
print("Done! Saving results...")
df.to_csv("submission.csv", index=False)
print(df)

Done! Saving results...
            id  luxury_score
0    634895718          78.2
1    299900595          93.4
2    962061404          99.6
3    887846414          70.3
4    227521863          85.8
..         ...           ...
868  321568250          88.0
869  294212716          76.4
870  807957764          92.1
871  213245302          69.6
872  792585463          98.9

[873 rows x 2 columns]
